In [1]:


from __future__ import annotations

import sys
from pathlib import Path
import pandas as pd
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import tensorflow as tf
import types


# -----------------------------------------------------------------------------
# Paths: allow importing both core package and WIP package when running from repo root
# -----------------------------------------------------------------------------
def change_to_repo_root(marker: str = "src") -> None:
    """Change CWD to the repository root (parent of `src`)."""
    here = Path.cwd()
    for parent in [here] + list(here.parents):
        if (parent / marker).is_dir():
            os.chdir(parent)
            break

change_to_repo_root("WIP")
ROOT = Path.cwd()
WIP_SRC = ROOT / "WIP" / "src"
CORE_SRC = ROOT / "src"
WIP= ROOT / "WIP"
sys.path.insert(0, str(WIP_SRC))
sys.path.insert(0, str(CORE_SRC))
sys.path.insert(0, str(WIP))

# -----------------------------------------------------------------------------
# Imports (WIP)
# -----------------------------------------------------------------------------
from Q_Sea_Battle_New.pyr_dataset_generation_utilities import generate_pyr_dataset
from Q_Sea_Battle_New.pyr_dataset_conversion_utilities import convert_internal_model_a
from Q_Sea_Battle_New.pyr_internal_model_a import PyrInternalModelA

# -----------------------------------------------------------------------------
# Imports (core)
# -----------------------------------------------------------------------------
from Q_Sea_Battle.game_layout import GameLayout



In [2]:

# -----------------------------------------------------------------------------
# Settings (as provided)
# -----------------------------------------------------------------------------
FIELD_SIZE = 4          # 4x4 -> N=16 (requires N a power of 2)
N2 = FIELD_SIZE * FIELD_SIZE
COMMS_SIZE = 1          # Pyramid requires 1 comm bit

P_HIGH = 1.0            # PR-assisted correlation parameter (stochastic mode, 2nd measurement)

GAMES_IN_EVAL_TOURNAMENT = 1000

SEED = 1234



layout_eval = GameLayout(
    field_size=FIELD_SIZE,
    comms_size=COMMS_SIZE,
    number_of_games_in_tournament=GAMES_IN_EVAL_TOURNAMENT,
    channel_noise=0.0,
    enemy_probability=0.5,
)
depth = 4

# --------------------------
# Generate canonical dataset (bits)
# --------------------------
NUM_GAMES = 150_000
ds_bits = generate_pyr_dataset(n2=N2, num_games=NUM_GAMES, seed=SEED, validate=True)

BETA = 10.0              
BATCH = 256
EPOCHS = 25

## Train model A

In [3]:
# # --- PURE-LOGIT compute_with_internal (Model A) ---
# def new_compute_with_internal_a(
#     self,
#     field_logits: tf.Tensor,
#     replay_out_a_logits_list: Optional[Sequence[tf.Tensor]] = None,
#     training: bool = False,
# ) -> Tuple[tf.Tensor, List[tf.Tensor], List[tf.Tensor]]:
#     x = tf.convert_to_tensor(field_logits, dtype=tf.float32)
#     if x.shape.rank != 2:
#         raise ValueError(f"field_logits must be rank-2 (B,n2); got {x.shape}.")
#     if x.shape[-1] is not None and int(x.shape[-1]) != self.n2:
#         raise ValueError(f"field_logits last dimension must be n2={self.n2}; got {x.shape[-1]}.")

#     if replay_out_a_logits_list is not None:
#         if not isinstance(replay_out_a_logits_list, (list, tuple)):
#             raise TypeError("replay_out_a_logits_list must be a Python list/tuple of tensors or None.")
#         if len(replay_out_a_logits_list) != self.depth:
#             raise ValueError(
#                 f"replay_out_a_logits_list must have length depth={self.depth}; got {len(replay_out_a_logits_list)}."
#             )

#     meas_list: List[tf.Tensor] = []
#     out_list: List[tf.Tensor] = []

#     state_logits = x
#     last_field_logits: tf.Tensor | None = None

#     for level in range(self.depth):
#         meas_layer = self.measure_layers[level]
#         comb_layer = self.combine_layers[level]
#         sr = self.sr_layers[level]

#         # Measurement logits (B, k_d)
#         try:
#             meas_logits = tf.cast(meas_layer(state_logits, training=training), tf.float32)
#         except TypeError:
#             meas_logits = tf.cast(meas_layer(state_logits), tf.float32)

#         zeros = tf.zeros_like(meas_logits)
#         first_flag = tf.ones((tf.shape(meas_logits)[0], 1), dtype=tf.float32)

#         inputs = {
#             "current_measurement": meas_logits,
#             "previous_measurement": zeros,
#             "previous_outcome": zeros,
#             "first_measurement": first_flag,
#         }

#         # Force replay outcome during supervised training (teacher out_a)
#         if replay_out_a_logits_list is not None:
#             replay_logits = tf.cast(tf.convert_to_tensor(replay_out_a_logits_list[level]), tf.float32)
#             tf.debugging.assert_equal(
#                 tf.shape(replay_logits)[-1],
#                 tf.shape(meas_logits)[-1],
#                 message=f"Replay outcome length mismatch at level {level}.",
#             )
#             inputs["replay_outcome_logits"] = replay_logits

#         out_logits = tf.cast(sr(inputs, training=training), tf.float32)

#         # Combine A: logits + logits -> next field logits
#         try:
#             next_field_logits = tf.cast(comb_layer(state_logits, out_logits, training=training), tf.float32)
#         except TypeError:
#             next_field_logits = tf.cast(comb_layer(state_logits, out_logits), tf.float32)

#         meas_list.append(meas_logits)
#         out_list.append(out_logits)

#         # PURE LOGIT feed-forward
#         state_logits = next_field_logits
#         last_field_logits = next_field_logits

#     if last_field_logits is None:
#         raise RuntimeError("Internal error: model depth produced no outputs.")

#     comm_logits = tf.cast(last_field_logits, tf.float32)  # (B,1) for Pyr
#     return comm_logits, meas_list, out_list

# --------------------------
# Convert to INTERNAL MODEL A view (PURE LOGITS)
#   X: field_logits, out_a_logits_list
#   Y: comm_target_bits, meas_in_a_bits_list
# --------------------------
field_logits_np, comm_target_np, meas_in_list_np, out_a_list_np = convert_internal_model_a(
    ds_bits,
    rep_field="hard_logit",
    rep_comm_target="bits",
    rep_meas_target="bits",
    rep_out_target="hard_logit",
    beta=BETA,
)

depth = len(meas_in_list_np)
assert depth == len(out_a_list_np)

# Convert numpy -> tensors
field_logits = tf.constant(field_logits_np, tf.float32)               # (N, n2)
comm_target = tf.constant(comm_target_np, tf.float32)                 # (N, 1) bits
meas_targets = [tf.constant(a, tf.float32) for a in meas_in_list_np]  # list of (N, k_d) bits
out_a_teacher = [tf.constant(a, tf.float32) for a in out_a_list_np]   # list of (N, k_d) logits

# IMPORTANT: use tuples, not lists, so tf.data keeps it as a nested structure
X = (field_logits, tuple(out_a_teacher))
Y = (comm_target, tuple(meas_targets))

tfds = tf.data.Dataset.from_tensor_slices((X, Y))
tfds = tfds.shuffle(200_000, seed=SEED, reshuffle_each_iteration=True)
tfds = tfds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

# --------------------------
# Instantiate model + patch compute_with_internal to pure-logit
# --------------------------
model_a = PyrInternalModelA(layout_eval, sr_mode="replay", beta=BETA, seed=SEED)
#model_a.compute_with_internal = types.MethodType(new_compute_with_internal_a, model_a)

bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
opt = tf.keras.optimizers.Adam(1e-3)

@tf.function
def comm_acc(y_bits, logits):
    pred = tf.cast(logits >= 0.0, tf.float32)
    return tf.reduce_mean(tf.cast(tf.equal(pred, y_bits), tf.float32))

@tf.function
def bit_acc(y_bits, logits):
    pred = tf.cast(logits >= 0.0, tf.float32)
    return tf.reduce_mean(tf.cast(tf.equal(pred, y_bits), tf.float32))

@tf.function
def train_step(x_field, x_out_tuple, y_comm, y_meas_tuple, comm_weight):
    with tf.GradientTape() as tape:
        comm_logits, meas_logits_list, out_logits = model_a.compute_with_internal(
            x_field,
            replay_out_a_logits_list=x_out_tuple,
            training=True,
        )
        # out_logits: list of tensors, x_out_tuple: tuple of tensors (teacher)
        per_level = [tf.reduce_mean(tf.abs(t - o)) for t, o in zip(x_out_tuple, out_logits)]
        delta_meas_out = tf.add_n(per_level) / tf.cast(len(per_level), tf.float32)
        # comm loss
        loss_comm = bce(y_comm, comm_logits)

        # per-level measurement loss
        loss_meas = 0.0
        for yt, yp in zip(y_meas_tuple, meas_logits_list):
            loss_meas += bce(yt, yp)
        loss_meas /= tf.cast(len(meas_logits_list), tf.float32)

        loss = comm_weight*loss_comm + (1.0)*loss_meas

    grads = tape.gradient(loss, model_a.trainable_variables)
    opt.apply_gradients([(g, v) for g, v in zip(grads, model_a.trainable_variables) if g is not None])

    acc = comm_acc(y_comm, comm_logits)
    accs = [bit_acc(yt, yp) for yt, yp in zip(y_meas_tuple, meas_logits_list)]
    meas_acc = tf.add_n(accs) / tf.cast(len(accs), tf.float32)
    return loss, loss_comm, loss_meas, acc, meas_acc, delta_meas_out

epoch = 0
print("Starting training... Stopping criterion: comm_acc > 0.9999 AND meas_acc > 0.9999")
while True:
    epoch += 1
    m_loss = tf.keras.metrics.Mean()
    m_comm = tf.keras.metrics.Mean()
    m_meas = tf.keras.metrics.Mean()
    m_acc  = tf.keras.metrics.Mean()
    m_delta = tf.keras.metrics.Mean()
    m_meas_acc = tf.keras.metrics.Mean()
    comm_weight = 0.1
    for (x_field, x_out_tuple), (y_comm, y_meas_tuple) in tfds:
        loss, lc, lm, acc, meas_acc,delta_meas_out = train_step(x_field, x_out_tuple, y_comm, y_meas_tuple, comm_weight)
        m_loss.update_state(loss)
        m_comm.update_state(lc)
        m_meas.update_state(lm)
        m_acc.update_state(acc)
        m_meas_acc.update_state(meas_acc)
        m_delta.update_state(delta_meas_out)    

    print(
        f"Epoch {epoch:02d}  "
        f"loss={m_loss.result().numpy():.4f}  "
        f"comm={m_comm.result().numpy():.4f}  "
        f"meas={m_meas.result().numpy():.4f}  "
        f"comm_acc={m_acc.result().numpy():.4f}  "
        f"delta_meas_out={m_delta.result().numpy():.4f}  "
        f"comm_weight={comm_weight:.2f}  "
        f"meas_acc={m_meas_acc.result().numpy():.4f}"
    )

    if m_acc.result().numpy() > 0.9999 and m_meas_acc.result().numpy() > 0.9999:
        print("Early stopping criterion met.")
        break

model_weights_dir=Path("WIP/weights_pyr_models")
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
filename_template=f"model_a_weights_{timestamp}.weights.h5"
model_a.save_weights_to(model_weights_dir / filename_template)
print(f"Model weights saved to {model_weights_dir / filename_template}")

Starting training... Stopping criterion: comm_acc > 0.9999 AND meas_acc > 0.9999
Epoch 01  loss=0.6963  comm=0.7125  meas=0.6251  comm_acc=0.4969  delta_meas_out=0.0000  comm_weight=0.10  meas_acc=0.5920
Epoch 02  loss=0.5529  comm=0.6940  meas=0.4835  comm_acc=0.5018  delta_meas_out=0.0000  comm_weight=0.10  meas_acc=0.6566
Epoch 03  loss=0.5162  comm=0.6939  meas=0.4469  comm_acc=0.5023  delta_meas_out=0.0000  comm_weight=0.10  meas_acc=0.6817
Epoch 04  loss=0.5038  comm=0.6941  meas=0.4343  comm_acc=0.4978  delta_meas_out=0.0000  comm_weight=0.10  meas_acc=0.6881
Epoch 05  loss=0.5016  comm=0.6939  meas=0.4322  comm_acc=0.5007  delta_meas_out=0.0000  comm_weight=0.10  meas_acc=0.6925
Epoch 06  loss=0.4620  comm=0.6364  meas=0.3984  comm_acc=0.5633  delta_meas_out=0.0000  comm_weight=0.10  meas_acc=0.7174
Epoch 07  loss=0.3327  comm=0.0028  meas=0.3324  comm_acc=0.9995  delta_meas_out=0.0000  comm_weight=0.10  meas_acc=0.7657
Epoch 08  loss=0.3045  comm=0.0003  meas=0.3045  comm_acc=